# 05 — HTML and vision preprocessing

HTML modality produces two representations:

| Output | Shape | Source |
|---|---|---|
| `html_vec` — 45 handcrafted features | `(B, 45)` | `extract_html_features` |
| `html_tokens` — token ids + mask | `(B, 512)` | `HTMLTextTokenizer` |

Vision modality is a screenshot tensor `(B, 3, 224, 224)`.

## Critical: masks must reflect reality

Availability is confounded with the label (phishing pages are more likely to still be
live). Never fabricate a modality you did not collect.

In [ ]:
import sys
from pathlib import Path

BACKEND = Path(r"E:\UIU\TRIMESTER - 11\Computer Security\CS_Project\Backend")
DATASET = BACKEND.parent / "Dataset"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

SEED = 42
np.random.seed(SEED)

print("backend :", BACKEND)
print("dataset :", DATASET)
print("numpy   :", np.__version__)
print("pandas  :", pd.__version__)

In [ ]:
def show(title, obj):
    print(f"\n=== {title} ===")
    print(obj)

def hr(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

NOTEBOOK_DIR = BACKEND / "notebooks"
ARTIFACT_DIR = NOTEBOOK_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def save_artifact(name, obj):
    """Persist a JSON-serialisable object and echo where it landed."""
    path = ARTIFACT_DIR / name
    path.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")
    print(f"saved -> {path}")
    return path

## 5.1 Manifest

In [ ]:
from pathlib import Path
from app.preprocessing.multimodal_dataset import load_manifest, MultiModalDataset

MANIFEST = BACKEND / "data" / "manifest.csv"
show("manifest", {"path": str(MANIFEST), "exists": MANIFEST.exists()})

mf = load_manifest(MANIFEST)
show("columns", list(mf.columns))
show("rows", len(mf))
print(mf.groupby("split")[["html_ok", "shot_ok"]].agg(["sum", "count"]))

### Availability confound — the number that decides if fusion is worth anything

In [ ]:
hr("availability by label")
show("html_ok rate", mf.groupby("label")["html_ok"].mean().round(4).to_dict())
show("shot_ok rate", mf.groupby("label")["shot_ok"].mean().round(4).to_dict())

gap = mf.groupby("label")["html_ok"].mean()
print(f"\ngap (phishing - legitimate) = {gap[1] - gap[0]:+.4f}")

## 5.2 Integrity check — manifest claims vs files on disk

`_read_html` returns `None` for a missing file instead of raising, and
`_image_tensor` returns zeros. So a deleted artifact silently becomes "modality
present in the mask, all-zero in the tensor". That is the one failure mode this
project is specifically designed to avoid, so verify it explicitly.

In [ ]:
html_missing = [p for p in mf.loc[mf.html_ok == True, "html_path"] if not Path(str(p)).is_file()]
shot_missing = [p for p in mf.loc[mf.shot_ok == True, "shot_path"] if not Path(str(p)).is_file()]

show("integrity", {
    "manifest_html_ok": int((mf.html_ok == True).sum()),
    "html_missing_on_disk": len(html_missing),
    "manifest_shot_ok": int((mf.shot_ok == True).sum()),
    "shot_missing_on_disk": len(shot_missing),
})
if html_missing:
    print("\nfix before reporting: set html_ok=False for these rows, or re-crawl")
    for p in html_missing[:10]:
        print("  missing:", p)

## 5.3 HTML features — 45 of them

In [ ]:
from app.preprocessing.html_features import HTML_FEATURE_NAMES, N_HTML_FEATURES, extract_html_features, extract_html_features_batch

print(f"N_HTML_FEATURES = {N_HTML_FEATURES}")
for i, n in enumerate(HTML_FEATURE_NAMES):
    print(f"  {i:>2}  {n}")

In [ ]:
SAMPLE_HTML = """<!doctype html><html><head><title>Secure Login</title>
<style>body{{color:red}}</style></head>
<body><form action="http://paypa1-secure.tk/collect" method="post">
<input type="text" name="user"><input type="password" name="pass">
<button>Sign in</button></form>
<script>alert('x')</script></body></html>"""

fv = extract_html_features(SAMPLE_HTML)
assert len(fv) == N_HTML_FEATURES
show("features for a credential-harvesting page",
     {n: round(v, 3) for n, v in zip(HTML_FEATURE_NAMES, fv) if v not in (0.0,)})

print("\nNone input (unavailable modality) must yield zeros, not crash:")
print("  ", extract_html_features(None)[:5], "...")

## 5.4 HTML tokeniser

`HTMLTextTokenizer.fit` is **not idempotent** — it appends to the existing vocabulary.
Call `fit` once on train text only; calling it twice inflates the vocab.

In [ ]:
from app.preprocessing.html_features import visible_text_of
from app.preprocessing.multimodal_dataset import _soup_of
from app.preprocessing.html_tokenizer import HTMLTextTokenizer

soup = _soup_of(SAMPLE_HTML)
text = visible_text_of(soup)
print("visible text:", repr(text[:200]))
print("note: visible_text_of DECOMPOSES <script>/<style> in the passed soup (destructive)")

In [ ]:
MAX_TOKENS = 512

train_html_docs = []
for p in mf.loc[(mf.split == "train") & (mf.html_ok == True), "html_path"].head(3000):
    try:
        train_html_docs.append(Path(str(p)).read_text(encoding="utf-8", errors="replace"))
    except OSError:
        continue
train_texts = [visible_text_of(_soup_of(d))[:4000] for d in train_html_docs]

htok = HTMLTextTokenizer(vocab_size=4096, max_tokens=MAX_TOKENS, min_count=2)
htok.fit(train_texts)

show("html tokenizer", {
    "vocab_size": htok.size, "pad_id": htok.pad_id, "unk_id": htok.unk_id,
    "max_tokens": MAX_TOKENS, "docs_used": len(train_texts),
})

before = htok.size
htok.fit(train_texts[:10])
print(f"\nsecond fit on 10 docs: vocab {before} -> {htok.size}")
print("""
HTMLTextTokenizer.fit APPENDS to `itos` rather than rebuilding it. Calling fit twice
on different corpora silently corrupts the id mapping. The count may not move when the
vocab already sits at its cap, so this is a correctness risk, not a size warning.
Call fit exactly once.""")

## 5.5 Vision tensors

In [ ]:
import torch
from app.preprocessing.multimodal_dataset import MultiModalDataset, MultiModalRow

shot_rows = mf[mf.shot_ok == True].head(64)

# Use the real MultiModalRow dataclass. A hand-rolled stand-in missing .mask()
# fails inside MultiModalDataset.__getitem__.
rows = [
    MultiModalRow(
        row_id=int(r["row_id"]),
        split=str(r["split"]),
        url=str(r["url"]),
        label=int(r["label"]),
        html=None,
        html_vec=np.zeros(N_HTML_FEATURES, dtype=np.float32),
        image_path=Path(str(r["shot_path"])),
    )
    for _, r in shot_rows.iterrows()
]

ds = MultiModalDataset(rows, image_size=224)
item = ds[0]
print("image tensor:", tuple(item["image"].shape), item["image"].dtype)
print("mask_vision :", float(item["mask_vision"]))
print("range       :", round(float(item["image"].min()), 4), "..", round(float(item["image"].max()), 4))

missing = ds._image_tensor(Path("does_not_exist.png"))
print("\nmissing file -> zeros tensor:", tuple(missing.shape),
      "all-zero:", bool((missing == 0).all()))
print("The mask is set by the CALLER, not by _image_tensor. This is the silent-gap risk.")

## 5.6 Collate

In [ ]:
from app.preprocessing.multimodal_dataset import collate_multimodal

batch = collate_multimodal([ds[0], ds[1], ds[2]])
for k, v in batch.items():
    print(f"  {k:<18} {tuple(v.shape)}")

## Next

`06_multimodal_fusion_training.ipynb` — HTML/vision encoders and availability-aware fusion.